In [ ]:
# oneclick_boot cell 1 — environment probe (generic; prints PROBE_JSON:<json>)
import json, os, platform, socket, subprocess, sys, time, urllib.error, urllib.request

def sh(cmd, timeout=45):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return (p.stdout + p.stderr).strip()[:4000]
    except Exception as e:
        return f"ERR {type(e).__name__}: {e}"

def egress(url, timeout=10):
    """Reachability probe: any HTTP response (even 4xx) counts as reachable."""
    t0 = time.time()
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "vesperlm-oneclick"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            r.read(64)  # connectivity only — do not pull the body
            return {"ok": True, "status": getattr(r, "status", "?"), "secs": round(time.time() - t0, 2)}
    except urllib.error.HTTPError as e:
        return {"ok": True, "status": e.code, "secs": round(time.time() - t0, 2)}
    except Exception as e:
        return {"ok": False, "err": f"{type(e).__name__}: {e}", "secs": round(time.time() - t0, 2)}

probe = {
    "hostname": socket.gethostname(),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "cpu_count": os.cpu_count(),
    "time_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
probe["meminfo"] = sh("free -h | head -2")
probe["disk"] = sh("df -h / /app /tmp 2>/dev/null")
probe["rocm_version_file"] = sh("cat /opt/rocm/.info/version 2>/dev/null")
probe["rocm_smi"] = sh("rocm-smi 2>&1 | head -80")
probe["rocm_smi_ids"] = sh("rocm-smi --showid 2>&1 | head -40")
probe["amd_smi"] = sh("amd-smi list 2>&1 | head -40")
try:
    import torch
    probe["torch"] = torch.__version__
    probe["torch_cuda"] = torch.version.cuda
    probe["hip"] = torch.version.hip
    probe["cuda_available"] = bool(torch.cuda.is_available())
    n = torch.cuda.device_count() if torch.cuda.is_available() else 0
    probe["gpu_count_torch"] = n
    probe["gpu_names"] = [torch.cuda.get_device_name(i) for i in range(n)]
    if n:
        probe["gpu_props_0"] = str(torch.cuda.get_device_properties(0))[:300]
except Exception as e:
    probe["torch_err"] = f"{type(e).__name__}: {e}"

import re
probe["gpu_count_rocm"] = len(re.findall(r"GPU\[\d+\]", probe.get("rocm_smi", "") + probe.get("rocm_smi_ids", "")))

probe["egress"] = {
    "github": egress("https://github.com/"),
    "raw_githubusercontent": egress("https://raw.githubusercontent.com/datacrystals/VesperLM/main/lab/runner.py"),
    "codeload": egress("https://codeload.github.com/datacrystals/VesperLM/tar.gz/refs/heads/main"),
    "huggingface": egress("https://huggingface.co/"),
    "hf_mirror": egress("https://hf-mirror.com/"),
    "pypi": egress("https://pypi.org/simple/"),
    "pypi_tuna": egress("https://pypi.tuna.tsinghua.edu.cn/simple/"),
}
probe["gpu_count"] = probe.get("gpu_count_torch") or probe.get("gpu_count_rocm") or 0
print("PROBE_JSON:" + json.dumps(probe, default=str))


In [ ]:
# oneclick_boot cell 2 — fetch job JSON from JOB_URL (orchestrator substitutes the
# placeholder below at exec time; generic fallback = committed smoke job on main)
import json, os, time, urllib.request

JOB_URL = "__ONECLICK_JOB_URL__"  # replaced by pod/oneclick_run.py --job-url
if JOB_URL.startswith("__"):
    JOB_URL = os.environ.get(
        "ONECLICK_JOB_URL",
        "https://raw.githubusercontent.com/datacrystals/VesperLM/main/lab/oneclick_jobs/smoke.json",
    )
    print(f"[cell2] placeholder not injected — falling back to {JOB_URL}")

job = None
last_err = None
for attempt in range(5):
    try:
        req = urllib.request.Request(JOB_URL, headers={"User-Agent": "vesperlm-oneclick"})
        with urllib.request.urlopen(req, timeout=30) as r:
            job = json.loads(r.read().decode())
        break
    except Exception as e:
        last_err = e
        print(f"[cell2] job fetch attempt {attempt + 1} failed: {type(e).__name__}: {e}")
        time.sleep(2 * (attempt + 1))
if job is None:
    raise RuntimeError(f"job fetch failed after 5 attempts: {last_err}")
for key in ("job_id", "repo_url", "ref", "workdir", "command", "result_file"):
    if key not in job:
        raise KeyError(f"job JSON missing required key: {key}")
print("JOB_JSON:" + json.dumps(job))


In [ ]:
# oneclick_boot cell 3 — clone VesperLM (git, codeload tarball fallback) + pip deps
import os, re, subprocess, sys

def sh(cmd, cwd=None, timeout=1200):
    print(f"[cell3] $ {cmd}")
    p = subprocess.run(cmd, shell=True, cwd=cwd, timeout=timeout)
    return p.returncode

REPO_DIR = None
for base in ("/app", "/tmp", os.path.expanduser("~")):
    try:
        probe = os.path.join(base, ".vesperlm_write_probe")
        open(probe, "w").write("x")
        os.remove(probe)
        REPO_DIR = os.path.join(base, "VesperLM")
        break
    except Exception:
        continue
if REPO_DIR is None:
    raise RuntimeError("no writable base dir for the clone")
print(f"[cell3] REPO_DIR={REPO_DIR}")

ref = job["ref"]
clone_cmd = (f"git clone --depth 1 --branch {ref} "
             f"{job['repo_url']} {REPO_DIR}")
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    rc = sh(f"git -C {REPO_DIR} fetch --depth 1 origin {ref} && "
            f"git -C {REPO_DIR} checkout -f {ref} && git -C {REPO_DIR} pull --ff-only")
    print(f"[cell3] reusing existing clone rc={rc}")
else:
    rc = sh(clone_cmd, timeout=600)
    if rc != 0:
        m = re.match(r"https://github\.com/([^/]+)/([^/.]+?)(?:\.git)?$", job["repo_url"])
        if not m:
            raise RuntimeError(f"cannot derive tarball URL from {job['repo_url']}")
        tar_url = f"https://codeload.github.com/{m.group(1)}/{m.group(2)}/tar.gz/{ref}"
        print(f"[cell3] git clone failed rc={rc}; falling back to tarball {tar_url}")
        rc = sh(f"mkdir -p {REPO_DIR} && "
                f"curl -L --retry 3 --retry-delay 3 -o /tmp/vesperlm.tgz {tar_url} && "
                f"tar xz -C {REPO_DIR} --strip-components=1 -f /tmp/vesperlm.tgz",
                timeout=600)
        if rc != 0:
            raise RuntimeError(f"repo fetch failed (git rc + tarball rc={rc})")

# Missing deps: probe imports, pip-install what is absent (image pins a pypi mirror).
missing = []
for mod, pipname in (("transformers", "transformers"),
                     ("numpy", "numpy"),
                     ("matplotlib", "matplotlib"),
                     ("fla", "flash-linear-attention")):
    try:
        __import__(mod)
    except Exception:
        missing.append(pipname)
if missing:
    pkgs = " ".join(missing)
    rc = sh(f"{sys.executable} -m pip install --no-input {pkgs}", timeout=900)
    if rc != 0:
        print(f"[cell3] mirror pip failed rc={rc}; retrying via pypi.org")
        sh(f"{sys.executable} -m pip install --no-input --index-url https://pypi.org/simple {pkgs}",
           timeout=900)
for mod in ("torch", "triton"):
    try:
        __import__(mod)
        print(f"[cell3] {mod} OK")
    except Exception as e:
        print(f"[cell3] WARNING required module {mod} missing: {type(e).__name__}: {e}")


In [ ]:
# oneclick_boot cell 4 — run job["command"] (cwd job["workdir"]) streaming stdout,
# with a bounded keepalive writing to container stdout (idle-reaper defense).
import datetime, os, signal, subprocess, threading, time

workdir = os.path.normpath(os.path.join(REPO_DIR, job["workdir"]))
os.makedirs(workdir, exist_ok=True)
env = dict(os.environ)
env["ONECLICK_REPO_DIR"] = REPO_DIR
for k, v in (job.get("env") or {}).items():
    env[k] = str(v)
stop_file = os.path.join(REPO_DIR, ".oneclick_stop")
if os.path.exists(stop_file):
    os.remove(stop_file)
ka_stop_at = time.time() + float(job.get("max_minutes", 45)) * 60 + 600

def keepalive():
    fd = None
    try:
        fd = open("/proc/1/fd/1", "a")
    except Exception as e:
        print(f"[cell4] container-stdout keepalive unavailable: {e}", flush=True)
    while time.time() < ka_stop_at and not os.path.exists(stop_file):
        line = f"[oneclick-keepalive] {datetime.datetime.now(datetime.timezone.utc).isoformat()}\n"
        try:
            if fd is not None:
                fd.write(line)
                fd.flush()
        except Exception:
            pass
        print(line.strip(), flush=True)
        for _ in range(24):
            if os.path.exists(stop_file) or time.time() >= ka_stop_at:
                break
            time.sleep(10)

threading.Thread(target=keepalive, daemon=True).start()
print(f"[cell4] workdir={workdir} command={job['command']}", flush=True)
p = subprocess.Popen(job["command"], shell=True, cwd=workdir, env=env,
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1, start_new_session=True)
try:
    for line in p.stdout:
        print(line, end="", flush=True)
    JOB_RC = p.wait()
except BaseException as e:
    print(f"\n[cell4] interrupted ({type(e).__name__}); killing job process group", flush=True)
    try:
        os.killpg(p.pid, signal.SIGKILL)
    except Exception:
        pass
    try:
        JOB_RC = p.wait(timeout=30)
    except Exception:
        JOB_RC = -9
    raise
finally:
    open(stop_file, "w").write(f"done rc={JOB_RC if 'JOB_RC' in dir() else 'unknown'}\n")
print(f"[cell4] job finished rc={JOB_RC}", flush=True)


In [ ]:
# oneclick_boot cell 5 — RESULT_JSON:<json> exfil via cell stdout (no credentials)
import json, os

result = {
    "job_id": job.get("job_id"),
    "job_rc": JOB_RC,
    "result_file": job["result_file"],
    "hostname": os.uname().nodename if hasattr(os, "uname") else None,
}
rp = os.path.join(REPO_DIR, job["result_file"])
try:
    with open(rp) as f:
        result["result"] = json.load(f)
    result["ok"] = True
except Exception as e:
    result["ok"] = False
    result["error"] = f"{type(e).__name__}: {e}"
print("RESULT_JSON:" + json.dumps(result, default=str))
